In [2]:
#импорт библиотек
import numpy as np
import pandas as pd
import re
import nltk
#import spacy
import string

In [3]:
#чтение датасета (отмена. Нам достаточно просто считать файл с помощью readlines() и записать это всё в словарь)
#df = pd.read_csv("/home/fblthp/PYNLP/lectures/PyLess/NLP/reviews.csv", encoding='UTF8', sep="\t")
#df.head()

file = open("/home/fblthp/PYNLP/NLP/AI/zamok.test","rt")
lines = file.readlines()
#sens = MySentences(lemmatized)
file.close()



In [4]:
from pymorphy3 import MorphAnalyzer
from nltk import sent_tokenize, word_tokenize, regexp_tokenize

from nltk.corpus import stopwords
STOPWORDS = set(stopwords.words('russian'))

def tokenize_lemmas(sent, pat=r"\b\w[`\w]+\b", morph=MorphAnalyzer()):
    lem = list()
    for tok in regexp_tokenize(sent, pat):
        if "`" in tok:
            lem.append(tok)
        else: lem.append(morph.parse(tok)[0].normal_form)
    return lem

lemmatized = list()
counter = [0,0]
skip_first = 94
skipped = list()

for i in range(0,len(lines)):
    lines[i] = lines[i].lower()
    x = tokenize_lemmas(lines[i])
    x = [word for word in x if word not in STOPWORDS]
    if len(x)<4 : continue
    if skip_first>0 and 'замо`к' in x: skip_first-=1; skipped.append(x);continue #для баланса вселенной
    if 'замо`к' in x: counter[0]+=1
    elif 'за`мок' in x: counter[1]+=1
    print(f'{i} {x}')
    lemmatized.append(x)
print(counter)
#print(len(lemmatized))
#print(lemmatized[760])

0 ['отвозить', 'собственный', 'за`мок', 'соблазнительный', 'суккуб', 'слуга']
33 ['отличный', 'возможность', 'получить', 'владение', 'нибыть', 'за`мок']
49 ['отправиться', 'за`мок', 'лев', 'сразу']
50 ['отправиться', 'портал', 'за`мок', 'единорог']
53 ['отправиться', 'королевский', 'за`мок', 'венсный']
58 ['отряд', 'пройти', 'сквозь', 'взбудоражить', 'за`мок', 'горячий', 'нож', 'сквозь', 'масло']
74 ['пара', 'штурмовать', 'феодальный', 'за`мок', 'англия']
78 ['первый', 'ваш', 'цель', 'за`мок', 'уйти']
79 ['первый', 'за`мок', 'называться', 'арсель']
80 ['первый', 'за`мок', 'сиять', 'нестерпимый', 'свет']
85 ['перенести', 'за`мок', 'немой', 'трясина']
87 ['перепрофилировать', 'за`мок', 'господин', 'лагран']
88 ['песчаный', 'за`мок', 'ливень', 'пляж']
106 ['хождение', 'каносса', 'понимать', 'путешествие', 'генрих', 'iv', 'за`мок', 'каносса']
110 ['подземный', 'владычица', 'вернуться', 'счастливый', 'за`мок']
114 ['подтверждать', 'право', 'собственность', 'за`мок']
117 ['позволить', 'разру

In [6]:
#sens.corpus[0]
lemmatized[0]

['отвозить', 'собственный', 'за`мок', 'соблазнительный', 'суккуб', 'слуга']

In [5]:
from gensim.models import Word2Vec
from gensim.models.word2vec import LineSentence

w2v_model = Word2Vec(
    min_count=3,
    window=2,
    vector_size=50,
    negative=6,
    alpha=0.03,
    min_alpha=0.0007,
    sample=6e-5,
    sg=1)

In [6]:
'''
from nltk.tokenize import sent_tokenize
from gensim.utils import simple_preprocess

class MySentences(object):
    def __init__(self, docs):
        self.corpus = docs
    def __iter__(self):
        for doc in self.corpus:
            doc_sentences = sent_tokenize(doc)
            for sent in doc_sentences:
                yield simple_preprocess(sent) # yields a tokenized 

sens = MySentences(lemmatized)
'''

w2v_model.build_vocab(lemmatized)
w2v_model.train(lemmatized, total_examples=w2v_model.corpus_count, epochs= 1, report_delay=1)

(58, 1914)

In [7]:
w2v_model.wv.most_similar(positive = ["за`мок"], negative = ['замо`к'], topn=25)

[('солдат', 0.38725730776786804),
 ('покинуть', 0.2787899971008301),
 ('вернуться', 0.25603926181793213),
 ('помощь', 0.2400689423084259),
 ('сильный', 0.21022067964076996),
 ('пара', 0.20141075551509857),
 ('номер', 0.18755103647708893),
 ('щёлкнуть', 0.17576484382152557),
 ('вход', 0.1651889681816101),
 ('навесный', 0.16104985773563385),
 ('песок', 0.14554961025714874),
 ('год', 0.14294806122779846),
 ('старинный', 0.13747631013393402),
 ('собираться', 0.13370458781719208),
 ('первый', 0.12313123792409897),
 ('спустя', 0.12311559170484543),
 ('мочь', 0.10804090648889542),
 ('словно', 0.10387574881315231),
 ('живот', 0.10355296730995178),
 ('свой', 0.09677908569574356),
 ('закрывать', 0.09444568306207657),
 ('давно', 0.09392712265253067),
 ('сунуть', 0.08976608514785767),
 ('твой', 0.08281761407852173),
 ('также', 0.07840597629547119)]

In [13]:
CBOW_model = Word2Vec(
    min_count=3,
    window=3,
    vector_size=50,
    negative=6,
    alpha=0.03,
    min_alpha=0.0007,
    sample=6e-5,
    sg=0)

CBOW_model.build_vocab(lemmatized)
CBOW_model.train(lemmatized, total_examples=w2v_model.corpus_count, epochs= 2, report_delay=1)


print(CBOW_model.wv.most_similar_to_given(CBOW_model.wv.most_similar(positive=["открыть","ключ"]), ["за`мок","замо`к"]))

'''
def mean_cosine(model, n=15):
    V = model.wv.vectors[:n]                       # the n most frequent words
    V = V / np.linalg.norm(V, axis=1, keepdims=True)
    C = V @ V.T
    return C[np.triu_indices(n, 1)].mean()

for name, m in [("CBOW", CBOW_model), ("skip-gram", w2v_model)]:
    print(name, " mean cosine:", round(mean_cosine(m), 4))
    print("   similar to 'за`мок':", [(w, round(s, 3)) for w, s in m.wv.similar_by_word('за`мок', topn=5)])
    print("   similar to 'замо`к':", [w for w in m.wv.similar_by_word('замо`к', topn=5)])
'''

KeyError: "Key '('палец', 0.3462159335613251)' not present"